<table>
    <tr> 
    <th>                                                                                   
        <table>
            <tr>                                                                                   
                 <th  style="font-size:150%;ftext-align:left;background-color:#053061;color:white;">Valeur</th>
                 <th  style="font-size:150%;text-align:left;background-color:#053061;color:white;">Nom classe</th>
             </tr>
            <tr>
                <th  style="font-size:150%;text-align:left">0</th>
                <th  style="font-size:150%;text-align:left">Arrière-Plan</th>
            </tr>
            <tr>
                <th  style="font-size:150%;text-align:left">1</th>
                <th  style="font-size:150%;text-align:left">Platelets</th>
            </tr>
            <tr>
                <th  style="font-size:150%;text-align:left">2</th>
                <th  style="font-size:150%;text-align:left">RBC</th>
            </tr>
            <tr>
                <th  style="font-size:150%;text-align:left">3</th>
                <th  style="font-size:150%;text-align:left">WBC</th>
            </tr>
        </table>
    </th>
    <th> 
        <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'>BloodMNIST<br> Standardized Biomedical Images</div>
        <div style='text-align: center'>
            <img src="https://raw.githubusercontent.com/rbizoi/FHU_TARGET_ScientificDays_2025/refs/heads/main/images/objet_detection_bcd.png" width="512">
        </div>      
    </th>
    </tr>
</table>

>> **site du projet** : https://public.roboflow.ai/object-detection/bccd

# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Initialisation du document</div></b>

In [ ]:
import os
# import json
# @param ["tensorflow", "jax", "torch"]
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['KERAS_BACKEND'] = 'tensorflow'  
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ['TF_CPP_MIN_LOG_LEVEL']='5'

import tensorflow as tf, tensorflow_models as tfm
import keras, keras_hub
tf.get_logger().setLevel('ERROR')

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:100%; border-radius:10px 10px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Import libriries </div></b>

In [2]:
import numpy as np, pandas as pd, seaborn as sns, warnings, os, sys, time, pprint, math, re 
import pprint as pp

from matplotlib import pyplot as plt

warnings.filterwarnings(action="ignore")

if int(str(sns.__version__).split('.')[1]) > 8 : 
    plt.style.use('seaborn-v0_8-darkgrid')
else:
    plt.style.use('seaborn-darkgrid')
    
sns.set(font_scale=2)

print(f"""La version des librairies utilisées :

Tensorflow : {tf.__version__}\tCUDA {tf.test.is_built_with_cuda()}\tGPU {tf.test.is_built_with_gpu_support()}\tXLA {tf.test.is_built_with_xla()}
Keras      : {keras.version()}
Pandas     : {pd.__version__}
NumPy      : {np.__version__}""")

La version des librairies utilisées :

Tensorflow : 2.19.1	CUDA True	GPU True	XLA True
Keras      : 3.12.0
Pandas     : 2.3.2
NumPy      : 1.26.4


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Outils du document</div></b>

In [3]:
sys.path.append(os.path.abspath('../outils/'))

In [4]:
if 'configProjet' in sys.modules.keys():  del(sys.modules['configProjet'])
if 'configModele' in sys.modules.keys():  del(sys.modules['configModele'])

In [5]:
from configProjet import palette, initParametresProjet 
from configProjet import afficheHistoriqueEntrainement, afficheProbabilites, afficheMatriceConfusion, afficheDataset, afficheDistributionsPipe

In [6]:
from configModele import modelDictionnaire, initParametresExecution, getPipelineDataset
from configModele import creationCompilationModele, creationRappelsExecution, entrainementModele
from configModele import sauvegarderModel, sauvegardeHistorique, executeApprentissageChoixClassifieurs
from configModele import modelCNNSimple, modelPersonalise, modelPersonaliseInception

I0000 00:00:1757446655.432948    6185 service.cc:152] XLA service 0x5ce9a315d680 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757446655.433005    6185 service.cc:160]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1757446655.520982    6185 service.cc:152] XLA service 0x5ce9a2f76c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757446655.520995    6185 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
I0000 00:00:1757446655.526227    6185 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22198 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Paramétrés de l’exécution et les flux de lecture des fichiers TFRecors</div></b>

In [7]:
image_size,batch_size,epochs,_,_,_,_ = \
       initParametresExecution(image_size=(256, 256, 3),
                            batch_size = 8,
                            epochs=1024,
                            repertoire='../donnees/BloodSegmented',
                            cycle_length=None,
                            deterministic=None,
                            repetition=1,
                            bufferAleatoire=1024)

In [8]:
tailleImageFichier = 416
image_size    = (256, 256, 3)
batch_size    = 8
epochs        = 128
nombreClasses = 3
cfg    = {'classes': {0: 'Arrière-Plan', 1: 'Platelets', 2: 'RBC', 3: 'WBC'}, 'couleurs': {0: '#ffffff', 1: '#be1229', 2: '#113d14', 3: '#752973'}}    
dictLabels = cfg

In [9]:
print(f"""
  image_size    = {image_size}
  batch_size    = {batch_size}
  epochs        = {epochs}
  nombreClasses = {nombreClasses}
  dictLabels    = {dictLabels}
       """)


  image_size    = (256, 256, 3)
  batch_size    = 8
  epochs        = 128
  nombreClasses = 3
  dictLabels    = {'classes': {0: 'Arrière-Plan', 1: 'Platelets', 2: 'RBC', 3: 'WBC'}, 'couleurs': {0: '#ffffff', 1: '#be1229', 2: '#113d14', 3: '#752973'}}
       


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Initialisation du projet</div></b>

In [10]:
nomModel = 'RetinanetResnetFPN'

In [11]:
repertoireEnregistrement,repertoireSauvegardes,repertoireModelCKP,repertoireModelSauvegarde,repertoireModelLogs = \
            initParametresProjet(nomProjet=f"5-bloodMNIST-{nomModel}-160x160-bs{batch_size:02d}-epochs{epochs:03d}",repertoireProjet='../ResultatsExecutions')

In [12]:
train_data_input_path = '/home/razvan/documentation/DeepLearning/donnees/BloodObjectDetection/apprentissage-00000-of-00001.tfrecord'
valid_data_input_path = '/home/razvan/documentation/DeepLearning/donnees/BloodObjectDetection/validation-00000-of-00001.tfrecord'
test_data_input_path = '/home/razvan/documentation/DeepLearning/donnees/BloodObjectDetection/test-00000-of-00001.tfrecord'
model_dir = f'{repertoireModelCKP}/'
export_dir =f'{repertoireModelSauvegarde}/'

# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Configuration du projet</div></b>

In [13]:
exp_config =  tfm.core.exp_factory.get_exp_config('retinanet_resnetfpn_coco')

In [14]:
num_classes = nombreClasses

HEIGHT, WIDTH = image_size[0],image_size[1]
IMG_SIZE = [HEIGHT, WIDTH, 3] #obligatoirement une liste

# Backbone config.
exp_config.task.freeze_backbone = False
exp_config.task.annotation_file = ''

# Model config.
exp_config.task.model.input_size = IMG_SIZE
exp_config.task.model.num_classes = num_classes + 1
exp_config.task.model.detection_generator.tflite_post_processing.max_classes_per_detection = exp_config.task.model.num_classes

# Training data config.
exp_config.task.train_data.input_path = train_data_input_path
exp_config.task.train_data.dtype = 'float32'
exp_config.task.train_data.global_batch_size = batch_size
exp_config.task.train_data.parser.aug_scale_max = 1.0
exp_config.task.train_data.parser.aug_scale_min = 1.0

# Validation data config.
exp_config.task.validation_data.input_path = valid_data_input_path
exp_config.task.validation_data.dtype = 'float32'
exp_config.task.validation_data.global_batch_size = batch_size

# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Initialisation des GPUs presents</div></b>

In [15]:
logical_device_names = [logical_device.name for logical_device in tf.config.list_logical_devices()]

if 'GPU' in ''.join(logical_device_names):
  print('This may be broken in Colab.')
  device = 'GPU'
elif 'TPU' in ''.join(logical_device_names):
  print('This may be broken in Colab.')
  device = 'TPU'
else:
  print('Running on CPU is slow, so only train for a few steps.')
  device = 'CPU'

# apprentissage = 756
# validation = 73
# bach_size = 8

train_steps = 96768*4
exp_config.trainer.steps_per_loop = 95 # steps_per_loop = num_of_training_examples // train_batch_size

exp_config.trainer.summary_interval = 95
exp_config.trainer.checkpoint_interval = 95*5
exp_config.trainer.validation_interval = 95*10
exp_config.trainer.validation_steps =  73 # validation_steps = num_of_validation_examples // eval_batch_size
exp_config.trainer.train_steps = train_steps
exp_config.trainer.optimizer_config.warmup.linear.warmup_steps = 100
exp_config.trainer.optimizer_config.learning_rate.type = 'cosine'
exp_config.trainer.optimizer_config.learning_rate.cosine.decay_steps = train_steps
exp_config.trainer.optimizer_config.learning_rate.cosine.initial_learning_rate = 0.1
exp_config.trainer.optimizer_config.warmup.linear.warmup_learning_rate = 0.05

This may be broken in Colab.


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Affichage des caractéristiques du projet</div></b>

In [16]:
pp.pprint(exp_config.as_dict())

{'runtime': {'all_reduce_alg': None,
             'batchnorm_spatial_persistent': False,
             'dataset_num_private_threads': None,
             'default_shard_dim': -1,
             'distribution_strategy': 'mirrored',
             'enable_xla': False,
             'gpu_thread_mode': None,
             'loss_scale': None,
             'mixed_precision_dtype': 'bfloat16',
             'num_cores_per_replica': 1,
             'num_gpus': 0,
             'num_packs': 1,
             'per_gpu_thread_count': 0,
             'run_eagerly': False,
             'task_index': -1,
             'tpu': None,
             'tpu_enable_xla_dynamic_padder': None,
             'use_tpu_mp_strategy': False,
             'worker_hosts': None},
 'task': {'allow_image_summary': False,
          'annotation_file': '',
          'differential_privacy_config': None,
          'export_config': {'cast_detection_classes_to_float': False,
                            'cast_num_detections_to_float': False,


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Création du modèle  dans le projet d’apprentissage</div></b>

In [17]:
if exp_config.runtime.mixed_precision_dtype == tf.float16:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

if 'GPU' in ''.join(logical_device_names):
  distribution_strategy = tf.distribute.MirroredStrategy()
elif 'TPU' in ''.join(logical_device_names):
  tf.tpu.experimental.initialize_tpu_system()
  tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='/device:TPU_SYSTEM:0')
  distribution_strategy = tf.distribute.experimental.TPUStrategy(tpu)
else:
  print('Warning: this will be really slow.')
  distribution_strategy = tf.distribute.OneDeviceStrategy(logical_device_names[0])

print('Done')

Done


In [18]:
with distribution_strategy.scope():
  task = tfm.core.task_factory.get_task(exp_config.task, logging_dir=model_dir)

# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Contrôle du jeu de données d’apprentissage</div></b>

In [19]:
for images, labels in task.build_inputs(exp_config.task.train_data).take(1):
  print()
  print(f'images.shape: {str(images.shape):16}  images.dtype: {images.dtype!r}')
  print(f'labels.keys: {labels.keys()}')


images.shape: (8, 256, 256, 3)  images.dtype: tf.float32
labels.keys: dict_keys(['cls_targets', 'box_targets', 'anchor_boxes', 'cls_weights', 'box_weights', 'image_info'])


# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Apprentissage</div></b>

In [20]:
model, eval_logs = tfm.core.train_lib.run_experiment(
    distribution_strategy=distribution_strategy,
    task=task,
    mode='train_and_eval',
    params=exp_config,
    model_dir=model_dir,
    run_post_eval=True)

restoring or initializing model...
restored model from ../ResultatsExecutions/5-bloodMNIST-RetinanetResnetFPN-160x160-bs08-epochs128/model.sauvegardes/checkpoints/ckpt-5275.
restored from checkpoint: ../ResultatsExecutions/5-bloodMNIST-RetinanetResnetFPN-160x160-bs08-epochs128/model.sauvegardes/checkpoints/ckpt-5275
train | step:   5275 | training until step 6225...


I0000 00:00:1757446666.264992    6403 cuda_dnn.cc:529] Loaded cuDNN version 91300


train | step:   5370 | steps/sec:   10.0 | output: 
    {'box_loss': 0.0018879977,
     'cls_loss': 0.18834026,
     'learning_rate': 0.09995252,
     'model_loss': 0.28274018,
     'total_loss': 0.65683126,
     'training_loss': 0.65683126}
train | step:   5465 | steps/sec:   23.9 | output: 
    {'box_loss': 0.0017956473,
     'cls_loss': 0.17811799,
     'learning_rate': 0.09995083,
     'model_loss': 0.26790032,
     'total_loss': 0.6364803,
     'training_loss': 0.6364803}
train | step:   5560 | steps/sec:   25.4 | output: 
    {'box_loss': 0.0018028915,
     'cls_loss': 0.16955797,
     'learning_rate': 0.0999491,
     'model_loss': 0.2597026,
     'total_loss': 0.6227209,
     'training_loss': 0.6227209}
train | step:   5655 | steps/sec:   24.9 | output: 
    {'box_loss': 0.0017473829,
     'cls_loss': 0.17130536,
     'learning_rate': 0.09994735,
     'model_loss': 0.25867447,
     'total_loss': 0.61617,
     'training_loss': 0.61617}
train | step:   5750 | steps/sec:   24.3 | o

# <b><div style='padding:18px;background-color:#d8dcd6;color:#030aa7;font-size:130%; border-radius:12px 12px; box-shadow: 8px 8px 8px #042b4c;text-align: left'>Sauvegarde du modèle à partir du projet pour inférences ultérieures</div></b>

In [21]:
tfm.vision.serving.export_saved_model_lib.export_inference_graph(
    input_type='image_tensor',
    batch_size=1,
    input_image_size=[HEIGHT, WIDTH],
    params=exp_config,
    checkpoint_path=tf.train.latest_checkpoint(model_dir),
    export_dir=export_dir)